# Exercise 8

In [ ]:
from functions import RandomnessTests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import random
random.seed(42)
import time
import tracemalloc
np.random.seed(42)
rng = np.random.default_rng(42)
from scipy.special import factorial
from scipy import stats
import heapq
from scipy.stats import chisquare
import pandas as pd

## Part 1

13. Let $X_1, \ldots, X_n$ be independent and identically distributed random variables having unknown mean $\mu$.  
For given constants $a < b$, we are interested in estimating

$
p = P\left\{ a < \frac{1}{n}\sum_{i=1}^n X_i - \mu < b \right\}.
$

a/ Explain how we can use the bootstrap approach to estimate $p$.

Vi ønsker at estimere sandsynligheden

$
p = P\left( a < \frac{1}{n}\sum_{i=1}^n X_i - \mu < b \right),
$

men den sande middelværdi $\mu $ er ukendt.  
Bootstrap‑metoden erstatter den ukendte fordeling med den empiriske fordeling baseret på de observerede data.

Bootstrap antager, at fordelingen af

$
\bar{X} - \mu
$

kan approximeres af bootstrap‑fordelingen

$
\bar{X}^* - \bar{X},
$

hvor $\bar{X}^*$ er gennemsnittet af en bootstrap‑stikprøve.



Beregn stikprøvegennemsnittet  
   $
   \bar{X} = \frac{1}{n}\sum_{i=1}^n X_i.
   $

For $ B $ bootstrap‑replikationer (fx $ B = 10\,000 $):
   - Træk en bootstrap‑stikprøve  
     $
     (X_1^*, \ldots, X_n^*) \sim \text{empirisk fordeling}
     $
     dvs. sample med tilbage‑lægning fra de observerede værdier.
   - Beregn bootstrap‑gennemsnittet  
     $
     \bar{X}^*.
     $
   - Beregn bootstrap‑statistikken  
     $
     T^* = \bar{X}^* - \bar{X}.
     $

3. Estimér sandsynligheden  
   $
   \hat{p} = \frac{1}{B}\sum_{i=1}^B \mathbf{1}\{ a < T_i^* < b \}.
   $

Dette giver et bootstrap‑estimat af den ønskede sandsynlighed $ p $.


b/ Estimate $p$ if $n = 10$ and the values of the $X_i$ are:

56, 101, 78, 67, 93, 87, 64, 72, 80, 69.

Take $a = -5$ and $b = 5$.

In [ ]:
random.seed(42)
np.random.seed(42)

X = np.array([56, 101, 78, 67, 93, 87, 64, 72, 80, 69])
n = len(X)
a, b = -5, 5
B = 10000
x_bar = X.mean()
T_star = np.empty(B)

for i in range(B):
    X_star = np.random.choice(X, size=n, replace=True)
    x_bar_star = X_star.mean()
    T_star[i] = x_bar_star - x_bar

p_hat = np.mean((T_star > a) & (T_star < b))

print("Bootstrap-estimat af p:", p_hat)

## Part 2

In the following three exercises, $(X_1, \ldots, X_n)$ is a sample from a distribution whose variance is the unknown $(\sigma^2)$.  
We plan to estimate $(\sigma^2)$ by the sample variance



$
S^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar{X})^2,
$



and we want to use the bootstrap technique to estimate $Var(S^2)$.

15. If $n = 15$ and the data are

5, 4, 9, 6, 21, 17, 11, 20, 7, 10, 21, 15, 13, 16, 8

approximate (by a simulation) the bootstrap estimate of $Var(S^2)$.

In [ ]:
random.seed(42)
np.random.seed(42)

X = np.array([5, 4, 9, 6, 21, 17, 11, 20, 7, 10, 21, 15, 13, 16, 8])
n = len(X)
B = 10000

def sample_variance(x):
    return np.sum((x - x.mean())**2) / (len(x) - 1)

S2 = sample_variance(X)

S2_star = np.empty(B)

for i in range(B):
    X_star = np.random.choice(X, size=n, replace=True)
    S2_star[i] = sample_variance(X_star)

Var_S2_hat = np.var(S2_star, ddof=1)

print("Bootstrap-estimat af Var(S^2):", Var_S2_hat)


## Part 3

In [ ]:
random.seed(42)
np.random.seed(42)

def bootstrap_median_variance(sample, k):
    sample = np.asarray(sample)
    n = len(sample)

    med = np.median(sample)

    med_star = np.empty(k)
    for i in range(k):
        boot_sample = np.random.choice(sample, size=n, replace=True)
        med_star[i] = np.median(boot_sample)

    var_boot = np.var(med_star, ddof=1)

    return med, var_boot

import numpy as np

def bootstrap_mean_variance(sample, k):
    sample = np.asarray(sample)
    n = len(sample)

    mean_original = np.mean(sample)

    mean_star = np.empty(k)
    for i in range(k):
        boot_sample = np.random.choice(sample, size=n, replace=True)
        mean_star[i] = np.mean(boot_sample)

    var_boot = np.var(mean_star, ddof=1)

    return mean_original, var_boot


def pareto_sample(n, beta=1, k=1.05, seed=123):
    U = np.random.random(n)
    return beta * (U ** (-1 / k))


### 1

In [ ]:
random.seed(42)
np.random.seed(42)

k = 100
n = 200
beta_P = 1
kp = 1.05

X = pareto_sample(n, beta=1, k=1.05)

print("Sample median:", np.median(X))
print("Sample mean:", np.mean(X))

### 2

In [ ]:
random.seed(42)
np.random.seed(42)
mean, var_mean = bootstrap_mean_variance(X, k=k)

### 3

In [ ]:
random.seed(42)
np.random.seed(42)
median, var_median = bootstrap_median_variance(X, k=k)

### 4

In [ ]:
random.seed(42)
np.random.seed(42)
print("Bootstrap Var(mean):", var_mean)
print("Bootstrap Var(median):", var_median)

### 5

Middelværdi er langt mere ustabil end medianen. Den høje bootstrap varians afspejler, at tunge haler (Pareto med k=1.05) gør gennemsnittet meget følsomt over for ekstreme observationer.

Medianen er robust. Den lave varians viser at den næsten ikke påvirkes af få meget store eller meget små værdier selvom fordelingen er ekstremt skæv